# SYSTÈME DE RÉSUMÉ AUTOMATIQUE - VERSION FINALE COMPLÈTE

Ce notebook est le rendu final de votre projet. Il intègre toutes les étapes demandées :
1. **Exploration des Données** (describe, info, columns)
2. **Nettoyage Professionnel** (regex, spaCy)
3. **Modélisation par Classe** (Summarizer)
4. **Sauvegarde et Export (Pickle)**
5. **Évaluation Métrique (Precision, Recall, F1 via ROUGE)**
6. **Traitement Global sur 11 000+ articles**

Contraintes : **Python + NLTK + spaCy**.

In [ ]:
!pip install nltk spacy pandas rouge-score scikit-learn tqdm
!python -m spacy download en_core_web_sm

## 1. Chargement et Exploration des Données

In [29]:
import pandas as pd
import os
import spacy
import nltk
from nltk.corpus import stopwords
from sklearn.feature_extraction.text import TfidfVectorizer
from rouge_score import rouge_scorer
from collections import Counter
from string import punctuation
import heapq
import numpy as np
import re
from tqdm import tqdm
import pickle

path_data = 'cnn_dailymail/test.csv'
if os.path.exists(path_data):
    df_full = pd.read_csv(path_data)
    print(f"Données chargées : {len(df_full)} articles.")
else:
    raise FileNotFoundError("Le fichier test.csv est manquant dans cnn_dailymail/")

print("\n--- COLONNES ---")
print(df_full.columns.tolist())

print("\n--- INFOS ---")
print(df_full.info())

print("\n--- DESCRIPTION ---")
print(df_full.describe())

Données chargées : 11490 articles.

--- COLONNES ---
['id', 'article', 'highlights']

--- INFOS ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11490 entries, 0 to 11489
Data columns (total 3 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   id          11490 non-null  object
 1   article     11490 non-null  object
 2   highlights  11490 non-null  object
dtypes: object(3)
memory usage: 269.4+ KB
None

--- DESCRIPTION ---
                                              id  \
count                                      11490   
unique                                     11490   
top     a6a5491edb0c96c4391b6a8c4504416b3572b3a1   
freq                                           1   

                                                  article  \
count                                               11490   
unique                                              11488   
top     Defiant Nigel Farage today insisted he did not...   
freq               

## 2. Nettoyage des Données

In [30]:
def nettoyer_texte(text):
    if not isinstance(text, str): return ""
    text = text.replace('\n', ' ')
    text = re.sub(' +', ' ', text)
    return text.strip()

print("Nettoyage en cours...")
tqdm.pandas()
df_sample = df_full.head(100).copy() # On travaille sur un échantillon pour l'évaluation rapide
df_sample['article_clean'] = df_sample['article'].progress_apply(nettoyer_texte)
df_sample['highlights_clean'] = df_sample['highlights'].progress_apply(nettoyer_texte)

Nettoyage en cours...


100%|██████████| 100/100 [00:00<00:00, 21368.98it/s]


## 3. Le Modèle (Classe Summarizer)

In [31]:
nlp_global = spacy.load('en_core_web_sm', disable=['ner', 'textcat'])
nltk.download('stopwords')
stop_words_global = set(stopwords.words('english'))

class SummarizerFinal:
    def __init__(self):
        self.nlp = nlp_global
        self.stop_words = stop_words_global
        
    def summarize(self, text, n=3):
        if not text or len(text) < 10: return ""
        doc = self.nlp(text)
        words = [t.text.lower() for t in doc if t.pos_ in ['NOUN', 'ADJ', 'VERB'] and not t.is_stop]
        freq = Counter(words)
        if not freq: return ""
        max_f = max(freq.values())
        for w in freq: freq[w] /= max_f
        scores = {sent: sum(freq.get(w.text.lower(), 0) for w in sent) for sent in doc.sents}
        res = heapq.nlargest(n, scores, key=scores.get)
        return " ".join([s.text for s in sorted(res, key=lambda x: x.start)])

model = SummarizerFinal()

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/mirahasina/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


## 4. Sauvegarde PKL (Pickle)
On exporte le modèle pour pouvoir le réutiliser sans le code source.

In [32]:
with open('modele_final.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Modèle sauvegardé dans 'modele_final.pkl'")

Modèle sauvegardé dans 'modele_final.pkl'


## 5. Évaluation des Performances (ROUGE Metrics)
Calcul de la Précision, du Rappel et du F1-Score sur l'échantillon.

In [33]:
scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)
metrics = []

print("Évaluation en cours sur 100 articles...")
for i, row in tqdm(df_sample.iterrows(), total=100):
    gen = model.summarize(row['article_clean'])
    s = scorer.score(row['highlights_clean'], gen)
    metrics.append({
        'Precision': s['rougeL'].precision,
        'Recall': s['rougeL'].recall,
        'F1-Score': s['rougeL'].fmeasure
    })

df_m = pd.DataFrame(metrics)
print("\n--- SCORES DE PERFORMANCE ---")
print(df_m.mean())

Évaluation en cours sur 100 articles...


100%|██████████| 100/100 [00:17<00:00,  5.74it/s]


--- SCORES DE PERFORMANCE ---
Precision    0.144026
Recall       0.291892
F1-Score     0.186939
dtype: float64


## 6. Traitement Global et Export final

In [34]:
print("Génération de TOUS les résumés (Attention : Long)... ")
df_full['resume_automatique'] = df_full['article'].progress_apply(model.summarize)
df_full[['article', 'highlights', 'resume_automatique']].to_csv('resultats.csv', index=False)
print("Exportation réussie dans 'resultats.csv'")

Génération de TOUS les résumés (Attention : Long)... 


100%|██████████| 11490/11490 [25:36<00:00,  7.48it/s] 


Exportation réussie dans 'resultats.csv'
